In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install cyvcf2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.8 MB/s eta 0:00:00


In [15]:
import pandas as pd
import numpy as np
import datetime
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from cyvcf2 import VCF
import regex as re

In [5]:
gnomad_path = "/content/drive/MyDrive/FYP_DATA/DATA/processed_data/gnomad/sas_filtered_normalized_final/gnomad_chr22_sas_normalized.vcf.gz"

In [14]:
def inspect_clinvar_vcf(vcf_path: str, n_variants: int = 10):
    print("="*80)
    print("="*80)

    vcf = VCF(vcf_path)

    # ---- HEADER INSPECTION ----
    print("\n📌 VCF HEADER:")
    print("-" * 80)

    for header_line in vcf.raw_header.strip().split("\n"):
        print(header_line)

    # ---- VARIANT INSPECTION ----
    print("\n" + "-"*80)
    print(f"📌 FIRST {n_variants} VARIANTS:")
    print("-" * 80)

    for i, variant in enumerate(vcf):
        print(
            f"{variant.CHROM}:{variant.POS}  "
            f"{variant.REF}→{variant.ALT[0] if variant.ALT else ''}  "
            f"INFO= {variant.INFO}"
            )

        if i + 1 >= n_variants:
            break


In [13]:
inspect_clinvar_vcf(gnomad_path)


📌 VCF HEADER:
--------------------------------------------------------------------------------
##fileformat=VCFv4.2
##FILTER=<ID=PASS,Description="All filters passed">
##hailversion=0.2.130-bea04d9c79b5
##FILTER=<ID=BOTH_FILTERED,Description="Failed variant filters in both exomes and genomes datasets. Refer to 'exomes_filters' and 'genomes_filters' within INFO for more information">
##FILTER=<ID=EXOMES_FILTERED,Description="Failed variant filters in the exomes dataset and either passed all variant filters in the genomes dataset or the variant was not present in the genomes dataset. Refer to 'exomes_filters' within INFO for more information">
##FILTER=<ID=GENOMES_FILTERED,Description="Failed variant filters in the genomes dataset and either passed all variant filters in the exomes dataset or the variant was not present in the exomes dataset. Refer to 'genomes_filters' within INFO for more information">
##INFO=<ID=AC_joint,Number=A,Type=Integer,Description="Alternate allele count in joi

In [17]:
def print_ac_joint_per_ancestry(gnomad_path: str, n_variants: int = 10):
    vcf = VCF(gnomad_path)

    EXCLUDE = {"XX", "XY", "raw", "joint", "exomes", "genomes", "grpmax", "remaining"}
    ancestries = set()
    for line in vcf.raw_header.strip().split("\n"):
        if not line.startswith("##INFO"):
            continue
        match = re.search(r'ID=AC_joint_([^,_]+)', line)
        if match:
            token = match.group(1)
            if token not in EXCLUDE:
                ancestries.add(token)
    ancestries = sorted(ancestries)

    print(f"Ancestries: {ancestries}\n")
    print("=" * 80)

    for i, variant in enumerate(vcf):
        print(f"{variant.CHROM}:{variant.POS}  {variant.REF}→{variant.ALT[0] if variant.ALT else ''}")
        for anc in ancestries:
            field = f"AC_joint_{anc}"
            val = variant.INFO.get(field)
            print(f"  {field}: {val}")
        print()
        if i + 1 >= n_variants:
            break

In [19]:
print_ac_joint_per_ancestry(gnomad_path, n_variants=10)

Ancestries: ['afr', 'ami', 'amr', 'asj', 'eas', 'fin', 'mid', 'nfe', 'sas']

chr22:10510355  AT→A
  AC_joint_afr: 3
  AC_joint_ami: 0
  AC_joint_amr: 0
  AC_joint_asj: 0
  AC_joint_eas: 7
  AC_joint_fin: 0
  AC_joint_mid: 0
  AC_joint_nfe: 12
  AC_joint_sas: 4

chr22:10510356  T→A
  AC_joint_afr: 1277
  AC_joint_ami: 56
  AC_joint_amr: 1441
  AC_joint_asj: 229
  AC_joint_eas: 1498
  AC_joint_fin: 244
  AC_joint_mid: 12
  AC_joint_nfe: 3750
  AC_joint_sas: 437

chr22:10510380  T→C
  AC_joint_afr: 0
  AC_joint_ami: 0
  AC_joint_amr: 0
  AC_joint_asj: 0
  AC_joint_eas: 0
  AC_joint_fin: 0
  AC_joint_mid: 0
  AC_joint_nfe: 0
  AC_joint_sas: 2

chr22:10510413  G→A
  AC_joint_afr: 0
  AC_joint_ami: 0
  AC_joint_amr: 0
  AC_joint_asj: 0
  AC_joint_eas: 0
  AC_joint_fin: 0
  AC_joint_mid: 0
  AC_joint_nfe: 0
  AC_joint_sas: 2

chr22:10510438  G→A
  AC_joint_afr: 0
  AC_joint_ami: 0
  AC_joint_amr: 0
  AC_joint_asj: 0
  AC_joint_eas: 6
  AC_joint_fin: 0
  AC_joint_mid: 0
  AC_joint_nfe: 0
  AC_

In [ ]:
input_gnomad = "/content/drive/MyDrive/FYP_DATA/DATA/processed_data/gnomad/sas_filtered_normalized_final/gnomad_chr22_sas_normalized.vcf.gz"
reference_fasta = "/content/drive/MyDrive/FYP_DATA/raw_data/reference_fasta/hg38_dna.fa"
output_dir = "/content/drive/MyDrive/FYP_DATA/DATA/processed_data/pure_sas_filtered_normalized_FINAL_VERSION"


In [20]:
gnomad_path = "/content/drive/MyDrive/FYP_DATA/DATA/processed_data/gnomad/pure_sas_filtered_normalized_FINAL_VERSION/gnomad_chr22_pure_sas.vcf.gz"

In [21]:
print_ac_joint_per_ancestry(gnomad_path, n_variants=10)

Ancestries: ['afr', 'ami', 'amr', 'asj', 'eas', 'fin', 'mid', 'nfe', 'sas']

chr22:10510380  T→C
  AC_joint_afr: 0
  AC_joint_ami: 0
  AC_joint_amr: 0
  AC_joint_asj: 0
  AC_joint_eas: 0
  AC_joint_fin: 0
  AC_joint_mid: 0
  AC_joint_nfe: 0
  AC_joint_sas: 2

chr22:10510413  G→A
  AC_joint_afr: 0
  AC_joint_ami: 0
  AC_joint_amr: 0
  AC_joint_asj: 0
  AC_joint_eas: 0
  AC_joint_fin: 0
  AC_joint_mid: 0
  AC_joint_nfe: 0
  AC_joint_sas: 2

chr22:10510545  C→A
  AC_joint_afr: 0
  AC_joint_ami: 0
  AC_joint_amr: 0
  AC_joint_asj: 0
  AC_joint_eas: 0
  AC_joint_fin: 0
  AC_joint_mid: 0
  AC_joint_nfe: 0
  AC_joint_sas: 1

chr22:10510917  G→A
  AC_joint_afr: 0
  AC_joint_ami: 0
  AC_joint_amr: 0
  AC_joint_asj: 0
  AC_joint_eas: 0
  AC_joint_fin: 0
  AC_joint_mid: 0
  AC_joint_nfe: 0
  AC_joint_sas: 1

chr22:10511378  GATAA→G
  AC_joint_afr: 0
  AC_joint_ami: 0
  AC_joint_amr: 0
  AC_joint_asj: 0
  AC_joint_eas: 0
  AC_joint_fin: 0
  AC_joint_mid: 0
  AC_joint_nfe: 0
  AC_joint_sas: 1

chr2

In [29]:
!apt-get install -y bcftools tabix

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libhts3 libhtscodecs2
Suggested packages:
  python3-numpy python3-matplotlib texlive-latex-recommended
The following NEW packages will be installed:
  bcftools libhts3 libhtscodecs2 tabix
0 upgraded, 4 newly installed, 0 to remove and 2 not upgraded.
Need to get 1,491 kB of archives.
After this operation, 4,626 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libhtscodecs2 amd64 1.1.1-3 [53.2 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libhts3 amd64 1.13+ds-2build1 [390 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 bcftools amd64 1.13-1 [697 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tabix amd64 1.13+ds-2build1 [351 kB]
Fetched 1,491 kB in 0s (5,973 kB/s)
Selecting previously unselected package libhtscodecs2:amd64.
(Reading da

In [32]:
import subprocess
import os

CHR = "21"

input_gnomad = f"/content/drive/MyDrive/FYP_DATA/DATA/processed_data/gnomad/sas_filtered_normalized_final/gnomad_chr{CHR}_sas_normalized.vcf.gz"
output_file  = f"/content/drive/MyDrive/FYP_DATA/DATA/processed_data/pure_sas_filtered_normalized_FINAL_VERSION/gnomad_chr{CHR}_pure_sas.vcf.gz"

# CHECK 1 — do the files actually exist?
print("Input exists :", os.path.exists(input_gnomad))
print("Output exists:", os.path.exists(output_file))

# CHECK 2 — try bcftools stats instead of piping
before_result = subprocess.run(
    f"bcftools stats {input_gnomad} | grep 'number of records'",
    shell=True, capture_output=True, text=True
)
print("Before raw output:", before_result.stdout)
print("Before stderr:", before_result.stderr)

after_result = subprocess.run(
    f"bcftools stats {output_file} | grep 'number of records'",
    shell=True, capture_output=True, text=True
)
print("After raw output:", after_result.stdout)
print("After stderr:", after_result.stderr)

Input exists : True
Output exists: True
Before raw output: #   number of records   .. number of data rows in the VCF
SN	0	number of records:	1512370

Before stderr: 
After raw output: #   number of records   .. number of data rows in the VCF
SN	0	number of records:	555107

After stderr: 
